In [14]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, atan2, sqrt
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from datetime import timedelta
from dateutil.relativedelta import relativedelta
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

from src.EqCat import EqCat
eqCat = EqCat()

In [ ]:
# Расположение исходного каталога землетрясений

dir_in = 'data'
file_in = 'NEIC_Global_1900_2025.csv'
catalogType = 'USGS'

file_path = 'data/All_boundaries'

In [ ]:
#Insert here your seismic source parameters [globalcmt.org]
#2011 Tohoku earthquake
earthquake = {
    'event': '2011 Tohoku earthquake',
    'time': '2011 03 11 05:47:32.8',
    'latitude': 37.52,
    'longitude': 143.05,
    'depth_km': 20,
    'Mw': 9.1,
    'scalar_moment_Nm': 5.31e29,
    'strike_deg': 203,
    'dip_deg': 10,
    'rake_deg': 88
}

In [ ]:
# Load the uploaded CSV file

df = pd.read_csv(f"{dir_in}/{file_in}")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286453 entries, 0 to 286452
Data columns (total 22 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   time             286453 non-null  object 
 1   latitude         286453 non-null  float64
 2   longitude        286453 non-null  float64
 3   depth            285781 non-null  float64
 4   mag              286453 non-null  float64
 5   magType          286452 non-null  object 
 6   nst              104523 non-null  float64
 7   gap              147791 non-null  float64
 8   dmin             77678 non-null   float64
 9   rms              238229 non-null  float64
 10  net              286453 non-null  object 
 11  id               286453 non-null  object 
 12  updated          286453 non-null  object 
 13  place            286453 non-null  object 
 14  type             286453 non-null  object 
 15  horizontalError  73431 non-null   float64
 16  depthError       151169 non-null  floa

In [ ]:
# Преобразования для региона

In [ ]:
latitudes = []
longitudes = []

with open(file_path, 'r') as file:
    for line in file:
        if not line.startswith(' '):
            continue  # skip comment lines without leading spaces
        try:
            parts = line.strip().split()
            if len(parts) != 2:
                print(f"Skipping malformed line: {line.strip()}")
                continue
            lat, lon = map(float, parts)
            latitudes.append(lat)
            longitudes.append(lon)
        except Exception as e:
            print(f"Error parsing line '{line.strip()}': {e}")

# Create DataFrame
boundaries_df = pd.DataFrame({
    'latitude': latitudes,
    'longitude': longitudes
})

In [ ]:
#Haversine formula for calculating distance between two points on Earth
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth radius in km
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c  # distance in km

In [ ]:
start_date = pd.to_datetime(earthquake['time']).tz_localize('UTC')
start_date -= relativedelta(days=1)
end_date = start_date + relativedelta(months=2)  #2 months time interval

print(f"Mainshock date: {start_date.date()} -> End date defining the time interval for aftershock identification: {end_date.date()}")

# Filter by time window
df_time = df[(df['time'] >= start_date) & (df['time'] <= end_date)].copy()

# Calculate Haversine distances
df_time['distance_km'] = df_time.apply(
    lambda row: haversine_distance(earthquake['latitude'], earthquake['longitude'], row['latitude'], row['longitude']),
    axis=1
)

# Filter by distance <= 500 km
filtered_df = df_time[df_time['distance_km'] <= 500].copy()

#Deleting events the provide zero distance to avoid log domain error
filtered_df.drop_duplicates(subset=['latitude', 'longitude', 'time'], inplace=True)

In [ ]:
eqCat.loadEqCat(f"{dir_in}/{file_in}", catalogType, usecols = (0,11,1,2,3,4))
print('total no. of events: ', eqCat.size())
print(sorted(eqCat.data.keys()))

total no. of events:  285781
['DY', 'Depth', 'HR', 'ID', 'Lat', 'Lon', 'MN', 'MO', 'Mag', 'SC', 'Time', 'YR']


In [ ]:
eqCat.saveMatBin(file_in.rsplit('.', 1)[0] + '.mat')
newEqCat = EqCat()
newEqCat.loadMatBin(file_in.rsplit('.', 1)[0] + '.mat')
print(newEqCat.size())
print(sorted(newEqCat.data.keys()))

285781
['DY', 'Depth', 'HR', 'ID', 'Lat', 'Lon', 'MN', 'MO', 'Mag', 'SC', 'Time', 'YR']
